In [ ]:
import numpy as np
import pandas as pd 

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies.head(2)

In [ ]:
movies.shape

In [ ]:
credits.head(2)

In [ ]:
movies = movies.merge(credits,on='title')
movies.head(1)

In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.head(2)

In [ ]:
import ast

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies.head(3)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head(3)

In [ ]:
movies['cast'] = movies['cast'].apply(convert)
movies.head(3)

In [ ]:
movies['cast'] = movies['cast'].apply(lambda x:x[0:3])
movies.head(3)

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)
movies.head(3)

In [ ]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

movies.head(3)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])
new.head(2)

In [ ]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000,stop_words='english')

vector = cv.fit_transform(new['tags']).toarray()

vector.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)

similarity

In [ ]:
new[new['title'] == 'The Lego Movie'].index[0]

In [ ]:
import difflib

def recommend(movie):
    movie = movie.lower()

    titles = new['title'].str.lower().tolist()

    close_match = difflib.get_close_matches(movie, titles)

    if not close_match:
        print("Movie not found.")
        return

    index = new[new['title'].str.lower() == close_match[0]].index[0]

    distances = sorted(
        list(enumerate(similarity[index])),
        reverse=True,
        key=lambda x: x[1]
    )

    for i in distances[1:6]:
        print(new.iloc[i[0]].title)

In [ ]:
recommend('avenger')

In [ ]:
vector.shapeimport pickle

pickle.dump(new, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("movie_list.pkl and similarity.pkl saved successfully!")